In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, count, avg

In [0]:
spark = SparkSession.builder.appName("course_tracker").getOrCreate()

In [0]:
students_data = [
    (1, "ranjitha", "ranjitha@gmail.com"),
    (2, "arun", "arun@gmail.com"),
    (3, "meena", "meena@gmail.com"),
    (4, "karthik", "karthik@gmail.com")
]

students_df = spark.createDataFrame(students_data, ["student_id", "name", "email"])


In [0]:
courses_data = [
    (1, "deep learning", "dr. kumar"),
    (2, "data science", "dr. priya"),
    (3, "cloud computing", "mr. ravi"),
    (4, "nlp", "dr. anitha")
]

courses_df = spark.createDataFrame(courses_data, ["course_id", "course_name", "instructor"])

In [0]:
enrollments_data = [
    (1, 1, 1, "2026-01-10"),
    (2, 1, 2, "2026-01-12"),
    (3, 2, 3, "2026-01-15"),
    (4, 3, 1, "2026-01-18"),
    (5, 4, 4, "2026-01-20")
]

enrollments_df = spark.createDataFrame(enrollments_data,
    ["enrollment_id", "student_id", "course_id", "enrollment_date"])

In [0]:
progress_data = [
    (1, 1, 40, "2026-02-01"),
    (2, 2, 75, "2026-02-05"),
    (3, 3, 20, "2026-02-03"),
    (4, 4, 110, "2026-02-10"),   # invalid
    (5, 5, None, "2026-02-12")   # missing
]

progress_df = spark.createDataFrame(progress_data,
    ["progress_id", "enrollment_id", "completion_percentage", "last_updated"])

In [0]:
students_df.show()
courses_df.show()
enrollments_df.show()
progress_df.show()

+----------+--------+------------------+
|student_id|    name|             email|
+----------+--------+------------------+
|         1|ranjitha|ranjitha@gmail.com|
|         2|    arun|    arun@gmail.com|
|         3|   meena|   meena@gmail.com|
|         4| karthik| karthik@gmail.com|
+----------+--------+------------------+

+---------+---------------+----------+
|course_id|    course_name|instructor|
+---------+---------------+----------+
|        1|  deep learning| dr. kumar|
|        2|   data science| dr. priya|
|        3|cloud computing|  mr. ravi|
|        4|            nlp|dr. anitha|
+---------+---------------+----------+

+-------------+----------+---------+---------------+
|enrollment_id|student_id|course_id|enrollment_date|
+-------------+----------+---------+---------------+
|            1|         1|        1|     2026-01-10|
|            2|         1|        2|     2026-01-12|
|            3|         2|        3|     2026-01-15|
|            4|         3|        1|    

In [0]:
progress_df = progress_df.fillna({"completion_percentage": 0})

progress_df = progress_df.withColumn(
    "completion_percentage",
    when(col("completion_percentage") > 100, 100)
    .when(col("completion_percentage") < 0, 0)
    .otherwise(col("completion_percentage"))
)

In [0]:
progress_df.show()

+-----------+-------------+---------------------+------------+
|progress_id|enrollment_id|completion_percentage|last_updated|
+-----------+-------------+---------------------+------------+
|          1|            1|                   40|  2026-02-01|
|          2|            2|                   75|  2026-02-05|
|          3|            3|                   20|  2026-02-03|
|          4|            4|                  100|  2026-02-10|
|          5|            5|                    0|  2026-02-12|
+-----------+-------------+---------------------+------------+



In [0]:
data_df = enrollments_df.join(progress_df, "enrollment_id").join(students_df, "student_id").join(courses_df, "course_id")

display(data_df)

course_id,student_id,enrollment_id,enrollment_date,progress_id,completion_percentage,last_updated,name,email,course_name,instructor
1,1,1,2026-01-10,1,40,2026-02-01,ranjitha,ranjitha@gmail.com,deep learning,dr. kumar
2,1,2,2026-01-12,2,75,2026-02-05,ranjitha,ranjitha@gmail.com,data science,dr. priya
3,2,3,2026-01-15,3,20,2026-02-03,arun,arun@gmail.com,cloud computing,mr. ravi
1,3,4,2026-01-18,4,100,2026-02-10,meena,meena@gmail.com,deep learning,dr. kumar
4,4,5,2026-01-20,5,0,2026-02-12,karthik,karthik@gmail.com,nlp,dr. anitha


In [0]:
course_summary = data_df.groupBy("course_name").agg(
    count("student_id").alias("total_students"),
    avg("completion_percentage").alias("avg_completion")
)

display(course_summary)

course_name,total_students,avg_completion
deep learning,2,70.0
data science,1,75.0
cloud computing,1,20.0
nlp,1,0.0


In [0]:
status_df = data_df.withColumn(
    "status",
    when(col("completion_percentage") >= 50, "completed")
    .otherwise("dropped")
)

status_summary = status_df.groupBy("course_name", "status").count()

display(status_summary)

course_name,status,count
deep learning,dropped,1
data science,completed,1
cloud computing,dropped,1
deep learning,completed,1
nlp,dropped,1


In [0]:
top_completed = status_df.filter(col("status") == "completed").groupBy("course_name").count().orderBy(col("count").desc())

display(top_completed)

course_name,count
data science,1
deep learning,1


In [0]:
final_df = data_df.select(
    "name",
    "course_name",
    "enrollment_date",
    "completion_percentage"
)

display(final_df)

name,course_name,enrollment_date,completion_percentage
ranjitha,deep learning,2026-01-10,40
ranjitha,data science,2026-01-12,75
arun,cloud computing,2026-01-15,20
meena,deep learning,2026-01-18,100
karthik,nlp,2026-01-20,0


In [0]:
final_df.write.format("delta").mode("overwrite").saveAsTable("final_details")

In [0]:
spark.sql("select * from final_details ").show()

+--------+---------------+---------------+---------------------+
|    name|    course_name|enrollment_date|completion_percentage|
+--------+---------------+---------------+---------------------+
|ranjitha|  deep learning|     2026-01-10|                   40|
|ranjitha|   data science|     2026-01-12|                   75|
|    arun|cloud computing|     2026-01-15|                   20|
|   meena|  deep learning|     2026-01-18|                  100|
| karthik|            nlp|     2026-01-20|                    0|
+--------+---------------+---------------+---------------------+

